In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer,TrainingArguments,Trainer
import wandb
wandb.login(key="97c0d73ce3f332743f73aed9bfd6ffb011ebafb4")
%matplotlib inline

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: jarif. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [2]:
! pip install transformers datasets evaluate accelerate

In [3]:
df=pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")

In [4]:
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [5]:
df.drop(columns=["id","keyword","location"],axis=1,inplace=True)

In [6]:
df.shape

(7613, 2)

In [7]:
df.isnull().sum()

text      0
target    0
dtype: int64

In [8]:
df.head()

,text,target
0,Our Deeds are the Reason of this #earthquake M...,1
1,Forest fire near La Ronge Sask. Canada,1
2,All residents asked to 'shelter in place' are ...,1
3,"13,000 people receive #wildfires evacuation or...",1
4,Just got sent this photo from Ruby #Alaska as ...,1


In [9]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [10]:
import evaluate

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions,labels=eval_pred
    predictions=np.argmax(predictions,axis=1)
    return accuracy.compute(predictions=predictions,references=labels)

**Before you start training your model, create a map of the expected ids to their labels with id2label and label2id:**

In [11]:
id2label = {0: "NOT_DISASTER", 1: "DISASTER"}
label2id = {"NOT_DISASTER": 0, "DISASTER": 1}

In [12]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased", num_labels=2, 
                                                           id2label=id2label, label2id=label2id)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
def preprocess_function(examples):
 
    tokenized = tokenizer(examples["text"], truncation=True,padding=True,max_length=512)
    
    tokenized["labels"] = examples["target"]
    return tokenized

In [14]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
dataset = Dataset.from_pandas(df)
train_test_split = dataset.train_test_split(test_size=0.2, seed=42)
dataset_dict = DatasetDict({
    "train": train_test_split["train"],
    "test": train_test_split["test"]
})

# Preprocess the datasets

In [15]:

tokenized_datasets = dataset_dict.map(preprocess_function,batched=True,remove_columns=dataset_dict["train"].column_names)

Map:   0%|          | 0/6090 [00:00<?, ? examples/s]

Map:   0%|          | 0/1523 [00:00<?, ? examples/s]

In [16]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6090
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1523
    })
})

In [17]:
training_args = TrainingArguments(
    output_dir="my_awesome_model",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch")



In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics)

trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.502200,0.539684,0.811556
2,0.431400,0.562799,0.835850
3,0.330000,0.904473,0.803020
4,0.217800,0.906320,0.820749
5,0.128500,1.013133,0.812869
6,0.114100,1.176468,0.822718
7,0.076400,1.231309,0.827315
8,0.062000,1.414409,0.797768
9,0.054800,1.480903,0.804334
10,0.051600,1.447275,0.809586


TrainOutput(global_step=15230, training_loss=0.20139861635995865, metrics={'train_runtime': 579.5089, 'train_samples_per_second': 105.089, 'train_steps_per_second': 26.281, 'total_flos': 1274788627048944.0, 'train_loss': 0.20139861635995865, 'epoch': 10.0})

In [19]:
test_df=pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

In [20]:
test_df.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [21]:
Id=test_df.id

In [22]:
test_df.drop(columns=["keyword","location","id"],axis=1,inplace=True)

In [23]:
test_df.head()

,text
0,Just happened a terrible car crash
1,"Heard about #earthquake is different cities, s..."
2,"there is a forest fire at spot pond, geese are..."
3,Apocalypse lighting. #Spokane #wildfires
4,Typhoon Soudelor kills 28 in China and Taiwan


In [24]:
from datasets import Dataset, DatasetDict

dataset_dict = DatasetDict({"valid": Dataset.from_dict(test_df)})

def preprocess_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset_dict.map(preprocess_function, batched=True, remove_columns=dataset_dict["valid"].column_names)


Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

In [25]:

tokenized_datasets

DatasetDict({
    valid: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 3263
    })
})

In [26]:
prediction = trainer.predict(tokenized_datasets["valid"])
y_pred = np.argmax(prediction.predictions, axis=1)


In [27]:
submission=pd.DataFrame({"id":Id,"target":y_pred})
submission.to_csv("sumission.csv",index=False)
submission.head()

,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1
